In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn import datasets

from ipywidgets import interact, IntSlider
from IPython.display import display, clear_output

In [3]:
cars = pd.read_csv('data_bonbanh/processed/processed_raw_cars.csv')
df = cars[['current_status','origin','price']].copy()
df['current_status'] = df['current_status'].map({'Xe đã dùng': 0, 'Xe mới': 1})
df['origin'] = df['origin'].map({'Lắp ráp trong nước': 0, 'Nhập khẩu': 1})

print(df.head(10))
print(df.shape)
print(type(df))

   current_status  origin  price
0               0       0    890
1               0       1   3680
2               0       1   1780
3               0       1   3090
4               1       0    819
5               0       1   2779
6               0       0    888
7               0       1   2880
8               0       1    345
9               1       1    610
(5784, 3)
<class 'pandas.core.frame.DataFrame'>


In [4]:
def prepare_kmeans_data(df, num_clusters):
    X = df[['price', 'current_status', 'origin']].copy()

    # Chuẩn hóa dữ liệu
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # Thực hiện K-Means
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    X['cluster'] = labels

    # Lấy tâm cụm ở giá trị gốc
    centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)

    return X, X_scaled, labels, centroids_original, scaler

def print_cluster_statistics(X, centroids_original, num_clusters):
    """In thống kê chi tiết của các cụm"""
    centroids_df = pd.DataFrame(centroids_original, columns=['price', 'current_status', 'origin'])
    centroids_df.index.name = 'Cluster'

    print(f"\n Kết quả  K-means với số cụm k={num_clusters} \n")
    print("Đặc điểm các phân khúc (tâm cụm):")
    print(centroids_df.round(2),'\n')

    # Thống kê từng cụm
    for i in range(num_clusters):
        cluster_data = X[X['cluster'] == i]
        print(f"Phân khúc {i}: n={len(cluster_data):>4} | "
              f"Giá TB: {cluster_data['price'].mean():>10.2f} | "
              f"% Mới: {cluster_data['current_status'].mean()*100:>5.1f}% | "
              f"% Nhập khẩu: {cluster_data['origin'].mean()*100:>5.1f}%")
    print()


def plot_3d_with_jitter(ax, X, labels, centroids_original, num_clusters, colors):
    """Vẽ biểu đồ 3D với jitter"""
    np.random.seed(42)
    for cluster_id in range(num_clusters):
        points = X[labels == cluster_id].copy()
        points['current_status_jitter'] = points['current_status'] + np.random.normal(0, 0.05, len(points))
        points['origin_jitter'] = points['origin'] + np.random.normal(0, 0.05, len(points))
        ax.scatter(points['price'], points['current_status_jitter'], points['origin_jitter'],
                   c=[colors[cluster_id]], s=30, alpha=0.6, label=f'PK {cluster_id}', edgecolors='black', linewidths=0.3)
    # Vẽ tâm cụm
    ax.scatter(centroids_original[:, 0], centroids_original[:, 1], centroids_original[:, 2], s=400, c='red', marker='*',
               edgecolors='black', linewidths=2, label='Tâm cụm', zorder=10)
    ax.set_xlabel("Giá xe", fontsize=9)
    ax.set_ylabel("Tình trạng", fontsize=9)
    ax.set_zlabel("Nguồn gốc", fontsize=9)
    ax.set_title(f"3D Plot với Jitter (K={num_clusters})", fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, loc='upper left')


def plot_pca_2d(ax, X_scaled, labels, num_clusters, colors):
    """Vẽ biểu đồ PCA 2D"""
    pca = PCA(n_components=2)
    principal_components = pca.fit_transform(X_scaled)

    for cluster_id in range(num_clusters):
        cluster_mask = labels == cluster_id
        ax.scatter(principal_components[cluster_mask, 0], principal_components[cluster_mask, 1],
                   c=[colors[cluster_id]], s=60, alpha=0.7, label=f'PK {cluster_id}', edgecolors='black', linewidths=0.3)
    explained_variance = pca.explained_variance_ratio_.sum()
    ax.set_title(f'PCA 2D (K={num_clusters})\nGiải thích {explained_variance*100:.1f}% phương sai', fontsize=11, fontweight='bold')
    ax.set_xlabel('PC1', fontsize=9)
    ax.set_ylabel('PC2', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)


def plot_price_boxplot(ax, X, num_clusters, colors):
    """Vẽ boxplot giá theo phân khúc"""
    box_data = [X[X['cluster'] == i]['price'].values for i in range(num_clusters)]
    bp = ax.boxplot(box_data, tick_labels=[f'PK {i}' for i in range(num_clusters)], patch_artist=True, showmeans=True)

    # Tô màu cho boxplot
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_xlabel('Phân khúc', fontsize=9)
    ax.set_ylabel('Giá xe', fontsize=9)
    ax.set_title(f'Phân bố Giá (K={num_clusters})', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')


def plot_price_vs_status(ax, X, num_clusters, colors):
    """Vẽ biểu đồ giá vs tình trạng"""
    for cluster_id in range(num_clusters):
        cluster_data = X[X['cluster'] == cluster_id]
        for status in [0, 1]:
            data = cluster_data[cluster_data['current_status'] == status]
            if len(data) > 0:
                position = cluster_id + (status - 0.5) * 0.25
                ax.scatter([position] * len(data), data['price'], c=[colors[cluster_id]], alpha=0.6,
                           s=40, marker='o' if status == 1 else 's', edgecolors='black', linewidths=0.3)

    ax.set_xlabel('Phân khúc', fontsize=9)
    ax.set_ylabel('Giá xe', fontsize=9)
    ax.set_title(f'Giá theo Tình trạng (K={num_clusters})\n○=Mới, □=Cũ', fontsize=11, fontweight='bold')
    ax.set_xticks(range(num_clusters))
    ax.set_xticklabels([f'PK {i}' for i in range(num_clusters)])
    ax.grid(True, alpha=0.3, axis='y')


def plot_price_vs_origin(ax, X, num_clusters, colors):
    """Vẽ biểu đồ giá vs nguồn gốc"""
    for cluster_id in range(num_clusters):
        cluster_data = X[X['cluster'] == cluster_id]
        for origin in [0, 1]:
            data = cluster_data[cluster_data['origin'] == origin]
            if len(data) > 0:
                position = cluster_id + (origin - 0.5) * 0.25
                ax.scatter([position] * len(data), data['price'], c=[colors[cluster_id]],
                           alpha=0.6, s=40, marker='^' if origin == 1 else 'v', edgecolors='black', linewidths=0.3)

    ax.set_xlabel('Phân khúc', fontsize=9)
    ax.set_ylabel('Giá xe', fontsize=9)
    ax.set_title(f'Giá theo Nguồn gốc (K={num_clusters})\n▲=Nhập khẩu, ▼=Lắp ráp', fontsize=11, fontweight='bold')
    ax.set_xticks(range(num_clusters))
    ax.set_xticklabels([f'PK {i}' for i in range(num_clusters)])
    ax.grid(True, alpha=0.3, axis='y')


def plot_summary_heatmap(ax, X, num_clusters):
    """Vẽ heatmap tổng hợp"""
    distribution = []
    for cluster_id in range(num_clusters):
        cluster_data = X[X['cluster'] == cluster_id]
        row = [
            len(cluster_data),
            cluster_data['price'].mean(),
            cluster_data['current_status'].mean() * 100,
            cluster_data['origin'].mean() * 100
        ]
        distribution.append(row)

    dist_df = pd.DataFrame(distribution,
                           columns=['Số lượng', 'Giá TB', '% Mới', '% NK'],
                           index=[f'PK {i}' for i in range(num_clusters)])

    sns.heatmap(dist_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
                cbar_kws={'label': 'Giá trị'}, linewidths=0.5)
    ax.set_title(f'Bảng Tổng hợp (K={num_clusters})', fontsize=11, fontweight='bold')
    ax.set_xlabel('')

def plot_kmeans_clusters(num_clusters=3):
    X, X_scaled, labels, centroids_original, scaler = prepare_kmeans_data(df, num_clusters)
    print_cluster_statistics(X, centroids_original, num_clusters)
    colors = sns.color_palette("tab10", n_colors=num_clusters)

    # 4. Tạo figure với 6 subplots
    fig = plt.figure(figsize=(24, 18))

    # Biểu đồ 1: 3D với Jitter
    ax1 = fig.add_subplot(2, 3, 1, projection='3d')
    plot_3d_with_jitter(ax1, X, labels, centroids_original, num_clusters, colors)

    # Biểu đồ 2: PCA 2D
    ax2 = fig.add_subplot(2, 3, 2)
    plot_pca_2d(ax2, X_scaled, labels, num_clusters, colors)

    # Biểu đồ 3: Boxplot
    ax3 = fig.add_subplot(2, 3, 3)
    plot_price_boxplot(ax3, X, num_clusters, colors)

    # Biểu đồ 4: Giá vs Tình trạng
    ax4 = fig.add_subplot(2, 3, 4)
    plot_price_vs_status(ax4, X, num_clusters, colors)

    # Biểu đồ 5: Giá vs Nguồn gốc
    ax5 = fig.add_subplot(2, 3, 5)
    plot_price_vs_origin(ax5, X, num_clusters, colors)

    # Biểu đồ 6: Heatmap
    ax6 = fig.add_subplot(2, 3, 6)
    plot_summary_heatmap(ax6, X, num_clusters)

    # Tiêu đề tổng thể
    plt.suptitle(f'PHÂN TÍCH K-MEANS CLUSTERING - {num_clusters} PHÂN KHÚC',
                fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

In [ ]:
# Create a slider for adjusting the number of clusters
num_clusters_slider = IntSlider (value=3, min=2, max=5,step=1 ,description='Number of Clusters', continuous_update=False)

#Create an interactive widget
interact (plot_kmeans_clusters, num_clusters = num_clusters_slider)